# Modalidade clínica: análise reproduzível

Este notebook preserva o arquivo anterior e o CSV. A classificação é **Healthy housemate versus casos incluídos no estudo**, sem confirmação presumida de PIF e sem alvo de virulência. Não há processamento de outra modalidade.

Execute as células em ordem. Instalação: `python -m pip install -r requirements_clinicos.txt`. Inicie Jupyter nesta pasta. As funções completas e suas docstrings estão em `clinica_pipeline.py`; justificativas detalhadas estão em `README.md`.


## 1. Configuração e bibliotecas

Centralize caminho, semente, folds e parâmetros. Hiperparâmetros e experimentos são pré-especificados no módulo. Não os altere após observar métricas sem tratar a nova análise como exploratória. O CSV é lido diretamente da pasta.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image
from clinica_pipeline import (
    Configuracao, iniciar_execucao, auditar_dados, criar_coorte,
    construir_atributos, analisar_coorte, associacoes_por_casa,
    criar_modelos, criar_pipeline, definir_folds, avaliar_modelos,
    finalizar_execucao, EXPERIMENTOS, ID, CASA, GRUPO, DIAGNOSTICO,
)

config = Configuracao()  # Para outro CSV: Configuracao(arquivo=r"C:\pasta\dados.csv")
display(pd.Series(vars(config), name="Configuração"))


## 2. Carregamento, esquema e proveniência

O carregamento preserva textos literais como NA. O esquema é validado; IDs ausentes/repetidos e casa ausente interrompem a análise. Cada execução cria uma pasta exclusiva, registrando versões, configurações e hashes do CSV e código.


In [ ]:
dados, pasta = iniciar_execucao(config)
print(f"Dimensões: {dados.shape}; saída: {pasta.resolve()}")
display(dados.head())
display(dados.dtypes.to_frame("Tipo na importação"))


## 3. Ausências, duplicados, categorias e inconsistências

Ausência semântica inclui NA/unk/unknown/vazio; não significa resultado negativo. Arquivos exportados preservam cada categoria original e a normalização proposta. Revise diferenças de caixa/espaços e a equivalência `not perfomed/not performed` (grafia), sem alterar a entrada. Tipos de origem são texto deliberadamente; as datas serão tipadas de modo explícito.


In [ ]:
auditoria = auditar_dados(dados, pasta)
display(auditoria)
display(dados.describe(include="all"))
display(pd.read_csv(pasta / "categorias_originais_e_normalizadas.csv", keep_default_na=False))
print("Duplicados exatos:", dados.duplicated().sum())
print("FIPID repetidos:", dados[ID].duplicated().sum())
display(dados.groupby(CASA).size().describe().to_frame("Animais por casa"))
display(Image(filename=str(pasta / "ausentes.png")))


## 4. Inclusão, exclusão e alvo conservador

Somente Include e Healthy housemate entram. Rótulos de exclusão são validados por lista explícita. A categoria diagnóstica precisa ser compatível com o grupo; novidades interrompem a execução para revisão. Very likely/Very Likely são equivalentes de grafia, sem converter suspeita em confirmação. Grupo e categoria diagnóstica não serão preditores.


In [ ]:
coorte = criar_coorte(dados, pasta)
display(coorte.groupby([GRUPO, DIAGNOSTICO]).size().rename("n"))
display(coorte.alvo.value_counts().rename(index={0: "Healthy housemate", 1: "Caso incluído"}))
print("Incluídos:", len(coorte), "Excluídos:", len(dados)-len(coorte))


## 5. Engenharia clínica e validação de datas

Idade exige nascimento e coleta válidos, em mês/dia/ano. Idades negativas, acima do limite de auditoria ou datas futuras tornam-se ausentes e são listadas para correção na fonte. Regras são determinísticas por registro, sem aprendizado no conjunto completo. Sexo/raça permanecem categorias; exames são apenas qualitativos, sem misturar títulos ou material. NA em forma clínica/tratamento não é interpretado como ausência de sinais/tratamento.


In [ ]:
atributos = construir_atributos(coorte, config, pasta)
assert atributos.index.equals(coorte.index)
display(atributos.head())
display(atributos.describe(include="all"))
display(pd.read_csv(pasta / "problemas_datas.csv"))
display(pd.read_csv(pasta / "ausentes_atributos.csv"))


## 6. Análise exploratória por grupo

As distribuições exportadas incluem contagem e percentual dentro do grupo. A idade usa somente datas válidas, sem imputar para a descrição. A ausência de forma clínica nos controles pode ser estrutural: observe-a antes de interpretar desempenho de modelos que usam essa informação.


In [ ]:
analisar_coorte(coorte, atributos, pasta)
for nome in ["classes", "idade_grupo", "distribuicao_sexo", "distribuicao_raca", "distribuicao_forma_clinica"]:
    display(Image(filename=str(pasta / f"{nome}.png")))
display(pd.read_csv(pasta / "idade_por_grupo.csv"))


## 7. Associações exploratórias e multiplicidade

Permutações restritas dentro da casa respeitam o agrupamento, condicionadas à hipótese de permutabilidade intradomiciliar sob a hipótese nula. Usa-se diferença média de idade e qui-quadrado como estatística para categorias, com p empírico (sem aproximação de grandes amostras). Ausência categórica entra explicitamente; associações podem refletir coleta. Casas sem contraste não informam o teste. BH considera a família de variáveis exportada, com as limitações de dependência entre testes descritas no README. Não há interpretação causal.


In [ ]:
associacoes = associacoes_por_casa(coorte, atributos, config, pasta)
display(associacoes)


## 8. Experimentos, pré-processamento e validação por casa

Demográfico é o experimento de referência. Forma, exames e tratamento são adicionados em experimentos retrospectivos separados, pois podem revelar o processo de diagnóstico/seleção. FIPID, casa, grupo e diagnóstico nunca entram como preditores. Imputação, OneHot e padronização são ajustados dentro de cada treino. A validação é estratificada aproximadamente por classe e agrupada por casa; todo fold deve conter as duas classes. Sem busca de hiperparâmetros ou escolha antecipada de vencedor.


In [ ]:
display(pd.Series(EXPERIMENTOS, name="Preditores por experimento"))
folds = definir_folds(coorte, config)
for numero, (treino, validacao) in enumerate(folds, 1):
    assert set(coorte.iloc[treino][CASA]).isdisjoint(coorte.iloc[validacao][CASA])
    print(numero, "treino:", len(treino), "validação:", len(validacao),
          "classes na validação:", coorte.iloc[validacao].alvo.value_counts().to_dict())
display(criar_pipeline(EXPERIMENTOS["demografico"], criar_modelos(config)["Regressao_logistica"]))


## 9. Treinamento, métricas e previsões out-of-fold

Dummy, regressão logística, SVM linear e Random Forest usam exatamente as mesmas partições. Cada animal tem uma previsão de um modelo que não o viu no treino, nem animais da mesma casa. AUC usa probabilidade ou margem SVM; nenhuma AUC mistura margens de folds diferentes. Média e desvio-padrão entre folds não representam avaliação externa nem intervalo de confiança. Precision/F1 sem positivos previstos recebem zero.


In [ ]:
resumo, previsoes, metricas = avaliar_modelos(coorte, atributos, config, pasta)
display(resumo)
for experimento in EXPERIMENTOS:
    display(Image(filename=str(pasta / f"comparacao_{experimento}.png")))


## 10. Erros e interpretação das variáveis

Revise falsos positivos e negativos junto às categorias de suspeita, sem alterar o alvo após observar os erros. Importância é queda de AUC por permutação nos dados de validação; não implica causalidade ou biomarcador. A perturbação pode romper correlações entre variáveis e entre animais da casa. O resumo de importância agrega primeiro repetições dentro do fold. Coeficientes por categoria OneHot são exportados separadamente, com categorias identificadas.


In [ ]:
display(previsoes.loc[previsoes.erro, [ID, CASA, DIAGNOSTICO, "alvo", "previsao", "experimento", "modelo", "fold"]])
display(pd.read_csv(pasta / "frequencia_erros_amostra.csv").sort_values("mean", ascending=False))
display(pd.read_csv(pasta / "importancia_validacao_resumo.csv"))
display(pd.read_csv(pasta / "coeficientes_categorias_por_fold.csv").head(20))
for modelo in config.modelos:
    display(Image(filename=str(pasta / f"matriz_demografico_{modelo}.png")))


## 11. Exportação, componente reutilizável e relatório

Salva a referência pré-especificada (logística demográfica) ajustada na coorte inteira para uso futuro; esse ajuste não tem avaliação independente. A interface retorna FIPID separado de previsão/probabilidade não calibrada. Não há outra modalidade nem combinação de probabilidades. Somente carregue joblib de origem confiável.


In [ ]:
finalizar_execucao(dados, coorte, atributos, resumo, metricas, associacoes, config, pasta)
display(Markdown((pasta / "RELATORIO.md").read_text(encoding="utf-8")))
print("Arquivos exportados:", len(list(pasta.iterdir())))
print(pasta.resolve())


## 12. Limitações e próximos requisitos científicos

Os resultados calculados são os exibidos acima e nos arquivos da execução. Hipóteses de efeito biológico precisam de confirmação independente. O dataset é pequeno, selecionado, tem faltantes diferenciais e rótulos de suspeita. Desempenho com informação diagnóstica pode apenas recuperar critérios de seleção; **não demonstra PIF confirmada, virulência ou utilidade clínica**.

Confirme o significado de Home number ID e se existem recoletas/animais com IDs distintos; a exclusão menciona uma recoleta. A unicidade de FIPID não resolve essa questão sozinha. Antes de avançar, obtenha dicionário de dados, protocolo diagnóstico, temporalidade dos exames, origem das datas e validação externa. Para comparar modelos com bases clínicas, proteômicas e genômicas independentes, documente proveniência, critérios diagnósticos, unidade amostral e validação de cada base.
